# ELO Rating Obtain

In [2]:
# First check if selenium and chromedriver are available
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time


def scrape_elo_selenium(year):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=options)
    driver.get(f"https://eloratings.net/{year}")

    # Wait for the table to render
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CLASS_NAME, "slick-row"))
    )
    time.sleep(2)  # let all rows load

    rows = driver.find_elements(By.CLASS_NAME, "slick-row")
    records = []
    for row in rows:
        cells = row.find_elements(By.CLASS_NAME, "slick-cell")
        if len(cells) >= 3:
            records.append(
                {"team_name_elo": cells[1].text.strip(), "elo": cells[2].text.strip()}
            )
    driver.quit()
    return pd.DataFrame(records)


# Test on one year first
df_test = scrape_elo_selenium(2006)
print(df_test.head(10))
print(df_test.shape)

  team_name_elo   elo
0        Brazil  2080
1        France  2060
2         Italy  2019
3   Netherlands  1989
4     Argentina  1980
5       Germany  1976
6       England  1960
7       Denmark  1902
8       Croatia  1896
9      Portugal  1896
(236, 2)


In [3]:
df_test

,team_name_elo,elo
0,Brazil,2080
1,France,2060
2,Italy,2019
3,Netherlands,1989
4,Argentina,1980
...,...,...
231,Guam,532
232,Niue,496
233,Cocos Islands,422
234,Palau,402


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time


def scrape_elo_year(driver, year):
    url = f"https://www.eloratings.net/{year}"
    print(f"Fetching {year}...")
    driver.get(url)
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # XPath equiv: /html/body/div[1]/div[2]/div[1]/div[6]
    target = soup.select(
        "body > div:nth-of-type(1) > div:nth-of-type(2) > div:nth-of-type(1) > div:nth-of-type(6)"
    )
    if not target:
        print(f"  WARNING: no data found for {year}")
        return pd.DataFrame()

    text = target[0].get_text(separator="\n")
    elements = [x.strip() for x in text.split("\n") if x.strip()]

    cols = [
        "rank",
        "team",
        "rating",
        "average_rank",
        "average_rating",
        "one_year_change_rank",
        "one_year_change_rating",
        "matches_total",
        "matches_home",
        "matches_away",
        "matches_neutral",
        "matches_wins",
        "matches_losses",
        "matches_draws",
        "goals_for",
        "goals_against",
    ]

    records = []
    n = len(cols)
    for i in range(0, len(elements) - n + 1, n):
        chunk = elements[i : i + n]
        if len(chunk) == n:
            records.append(dict(zip(cols, chunk)))

    df = pd.DataFrame(records)
    df["year"] = year
    return df


# Run for your 5 tournament years
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

YEARS = [2005, 2009, 2013, 2017, 2021]  # end-of-year before each tournament
records = []
for year in YEARS:
    df = scrape_elo_year(driver, year)
    records.append(df)
    time.sleep(3)

driver.quit()

elo_raw = pd.concat(records, ignore_index=True)
print(elo_raw.shape)
print(elo_raw.head(10))

Fetching 2005...
Fetching 2009...
Fetching 2013...
Fetching 2017...
Fetching 2021...
(1045, 17)
   rank      team rating average_rank average_rating one_year_change_rank  \
0     1    Brazil   2027            5           1981                   +4   
1  1992        17   1820           +5            +53                  653   
2  1757        −1    −12          672            340                  259   
3   −18       662    267          323             72                  321   
4   389       449     77          531            176                  208   
5    65       271    108          133              7                Italy   
6   130       171      8    Argentina           1931                    5   
7     9  Portugal   1910           21           1762                    0   
8  1903        15   1794           +3            +31                  864   
9  1892        +1    +10          778            333                  328   

  one_year_change_rating matches_total matches_home matc

In [6]:
elo_raw.to_csv("elo_ratings.csv", index=False)

***need to fix this scraper